# m25 lab · Conduction velocity estimation

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Conduction velocity is the inverse of the activation-time gradient. You see how noise in the gradient biases speed, how timing error becomes speed error for two points and for triangles, how a local plane fit trades noise for bias, how a clique of electrodes estimates velocity from potentials without activation times, and how these estimators perform on a map with a slow region.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Speed from a gradient | eq-m25-gradient |
| B | Two points and triangles | eq-m25-pair, eq-m25-triangle |
| C | Local plane fits | eq-m25-planefit |
| D | Velocity from a clique | eq-m25-omnipolar |
| E | A conduction velocity map | all |

Times in ms, lengths in mm, speeds in mm/ms (= m/s). Speeds of 1.0 and 0.6 m/s follow Heil and colleagues; timing errors, point counts and geometry are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import velocity as VE, mapping as MP, meshes as M
from scipy.spatial import Delaunay

## Part A · Speed from a gradient

The velocity is $\nabla T/\lvert\nabla T\rvert^2$ and the speed $1/\lvert\nabla T\rvert$ (eq-m25-gradient). Noise in the
gradient adds to its magnitude on average, $\mathrm E\lvert\hat g\rvert^2 = \lvert g\rvert^2 + 2\sigma_g^2$ in 2D, so speeds
read low; and because speed is an inverse, fast conduction (a small gradient) is hit hardest.

In [ ]:
rng = np.random.default_rng(25)
sg = 0.2                                                 # gradient noise per component (ms/mm), illustrative
for c in (0.3, 0.6, 1.0, 2.0):
    g = np.array([1 / c, 0.0]) + rng.normal(0, sg, (100_000, 2))
    s = VE.speed_from_gradient(g)
    print(f"true {c:.1f} m/s: median estimate {np.median(s):.2f}, 10-90% range {np.percentile(s, 10):.2f}-{np.percentile(s, 90):.2f} m/s")

In [ ]:
# Checks
g = np.array([1.0, 0.0]) + rng.normal(0, sg, (100_000, 2))
assert abs(np.sqrt(np.mean(np.sum(g**2, 1))) - VE.inflated_gradient_rms(1.0, sg)) < 0.01

## Part B · Two points and triangles

Two points $h$ apart along propagation, each timed with SD $\sigma$, give a relative speed SD of $\sqrt2\,\sigma c/h$
(eq-m25-pair). Three points give a gradient with covariance $\sigma^2E^{-1}SE^{-\top}$, where $E$ holds two edges and
$S = \begin{psmallmatrix}2&1\\1&2\end{psmallmatrix}$ (eq-m25-triangle): an equilateral triangle matches the pair, while a
thin sliver is far worse across its short side.

In [ ]:
for h in (2.0, 4.0, 8.0):
    print(f"h = {h} mm, sigma = 1 ms, 1.0 m/s: relative SD {100 * VE.pair_relative_error(1.0, 1.0, h):.0f}%")
for name, P in (("equilateral", VE.equilateral(4.0)), ("sliver", np.array([[0, 0], [4, 0], [2, 0.4]], float))):
    C = VE.triangle_covariance(P, 1.0)
    print(f"{name:11s}: gradient SDs {np.sqrt(np.linalg.eigvalsh(C)).round(3)} ms/mm")

In [ ]:
# Checks
assert np.allclose(VE.triangle_covariance(VE.equilateral(4.0), 1.0), 2 / 16 * np.eye(2))

## Part C · Local plane fits

Fitting $T = t_0 + \mathbf g\cdot\mathbf x$ to the $n$ points within a radius $R$ (Bayly and colleagues used polynomial
surfaces) averages noise: the gradient covariance is $\sigma^2(X_c^\top X_c)^{-1}$, about $(2\sigma/(R\sqrt n))^2$ per
component for a disk (eq-m25-planefit). But a large radius also averages over curved activation, which biases the
speed near a focal source. Find the radius that balances the two.

In [ ]:
src = np.array([0.0, 0.0]); c = 1.0; sigma = 1.0
center = np.array([10.0, 0.0])                         # 10 mm from a focal source
pts = rng.uniform(-20, 20, (2000, 2)) + center          # dense illustrative sampling (about 1.25 per mm^2)
print(" R (mm)   n   bias (m/s)   SD (m/s)   RMS (m/s)")
for R in (1.5, 2.5, 4.0, 6.0, 9.0):
    m = np.linalg.norm(pts - center, axis=1) <= R
    P = pts[m]
    est = []
    for _ in range(300):
        T = np.linalg.norm(P - src, axis=1) / c + rng.normal(0, sigma, len(P))
        g, _ = VE.plane_fit(P, T)
        est.append(1 / np.linalg.norm(g))
    est = np.array(est)
    print(f"{R:6.1f} {m.sum():4d} {np.mean(est) - c:+10.3f} {np.std(est):10.3f} {np.sqrt(np.mean((est - c) ** 2)):10.3f}")

In [ ]:
# Checks
P = pts[np.linalg.norm(pts - center, axis=1) <= 4.0]
assert abs(np.sqrt(VE.plane_fit_covariance(P, 1.0)[0, 0]) - VE.disk_fit_sd(1.0, len(P), 4.0)) < 0.2 * VE.disk_fit_sd(1.0, len(P), 4.0)

## Part D · Velocity from a clique

For a plane wave $\phi(\mathbf x, t) = U(\hat n\cdot\mathbf x - ct)$ of any shape, $\mathbf v = -\phi_t\nabla\phi/\lvert\nabla\phi\rvert^2$
is exact (eq-m25-omnipolar): no activation times needed. A clique estimates $\nabla\phi$ by differences between
electrodes $d$ apart. For the cross-shaped clique here, at the front's arrival along an axis, the speed reads
$c(1 + (d/2)^2/a^2)$: fast, more so for wide cliques and electrodes close to the tissue. Other stencils have other
biases.

In [ ]:
for h in (0.5, 1.0, 2.0):
    a = 0.3 + h
    row = [np.linalg.norm(VE.clique_velocity(1.0, 0.0, a, d)) for d in (0.5, 1.0, 2.0, 4.0)]
    print(f"height {h} mm: speed ratio at d = 0.5, 1, 2, 4 mm: " + ", ".join(f"{r:.2f}" for r in row))
for th in (10, 25, 40):
    v = VE.clique_velocity(1.0, np.radians(th), 1.3, 2.0)
    print(f"angle {th} deg: direction estimate {np.degrees(np.arctan2(v[1], v[0])):.1f} deg, speed ratio {np.linalg.norm(v):.2f}")

In [ ]:
# Checks
assert abs(np.linalg.norm(VE.clique_velocity(1.0, 0.0, 1.3, 2.0)) - VE.clique_speed_ratio_axis(2.0, 1.3)) < 1e-6

## Part E · A conduction velocity map

A 40-mm patch is activated from one corner at 1.0 m/s, with a 7-mm-radius region conducting at 0.6 m/s (the values
Heil and colleagues reported for normal- and low-voltage left atrium). Sample 200 sites with 1-ms timing error and
estimate speed per triangle of a Delaunay triangulation, and by plane fits within 5 mm.

In [ ]:
V, F = M.lattice_mesh(41, 1.0, jitter=0.2, seed=25)
slow_c = np.array([22.0, 17.0])
slow = np.linalg.norm(V[F].mean(1)[:, :2] - slow_c, axis=1) < 7
T = MP.eikonal(V, F, np.argmin(np.linalg.norm(V[:, :2] - 2.0, axis=1)), np.where(slow, 0.6, 1.0))
idx = rng.choice(len(V), 200, replace=False)
P, Tn = V[idx, :2], T[idx] + rng.normal(0, 1.0, 200)
tri = Delaunay(P)
spd_tri = np.array([1 / np.linalg.norm(VE.triangle_gradient(P[s], Tn[s])) for s in tri.simplices])
tc = P[tri.simplices].mean(1)
G = np.stack(np.meshgrid(np.arange(4, 37, 2.0), np.arange(4, 31, 2.0)), -1).reshape(-1, 2)
spd_fit = VE.local_plane_speeds(P, Tn, G, 5.0)
def summary(xy, s):
    ins = np.linalg.norm(xy - slow_c, axis=1) < 5
    out = (~ins) & (np.linalg.norm(xy - 2.0, axis=1) > 8) & (np.linalg.norm(xy - slow_c, axis=1) > 9)
    return np.nanmedian(s[ins]), np.nanmedian(s[out]), np.nanpercentile(s[out], [25, 75])
for name, xy, s in (("triangles", tc, spd_tri), ("plane fit, 5 mm", G, spd_fit)):
    i, o, q = summary(xy, s)
    print(f"{name:16s}: slow region median {i:.2f} m/s (true 0.6); elsewhere median {o:.2f} m/s, IQR {q[0]:.2f}-{q[1]:.2f} (true 1.0)")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].tripcolor(P[:, 0], P[:, 1], tri.simplices, facecolors=np.clip(spd_tri, 0, 2), cmap="viridis", vmin=0, vmax=2); ax[0].set_title("triangles")
sc = ax[1].scatter(G[:, 0], G[:, 1], c=np.clip(spd_fit, 0, 2), cmap="viridis", vmin=0, vmax=2, s=40, marker="s"); ax[1].set_title("plane fit, 5 mm")
for a_ in ax: a_.add_patch(plt.Circle(slow_c, 7, fill=False, ls="--")); a_.set_aspect("equal")
fig.colorbar(sc, ax=ax, label="speed (m/s)"); plt.show()

In [ ]:
# Checks
q_tri = summary(tc, spd_tri)[2]; q_fit = summary(G, spd_fit)[2]
assert (q_fit[1] - q_fit[0]) < (q_tri[1] - q_tri[0])

## Where to go next

- Weight the plane fit by distance, or fit a polynomial surface as Bayly and colleagues did, and compare the bias.
- Use the window and reference errors of m22 as the timing error and see which estimator survives.
- m26 turns these errors into an uncertainty map.